# 01 - Deploy Infrastructure

This notebook deploys Azure resources using the Bicep template in `infra/`, then writes `env/.env` automatically for downstream notebooks.

**Estimated time:** 10-20 minutes

---

## What gets deployed
- Azure AI Services account (Foundry-aligned `kind: AIServices`)
- GPT-4o model deployment used by downstream notebooks

---

## Step 1 - Set deployment variables

In [ ]:
BICEP_RESOURCE_GROUP = "rg-gpt4o01-feasibility"
ARM_RESOURCE_GROUP = "rg-gpt4o01-feasibility-arm"
LOCATION = "swedencentral"
DEMO_ID = "gpt4o01"
DEPLOYMENT_RUN_NAME = "gpt4o01-infra-arm-test"  # change per run if needed
INFRA_TEMPLATE_KIND = "arm"  # options: "bicep" or "arm"

template_file_map = {
    "bicep": "../infra/main.bicep",
    "arm": "../infra/main.json",
}

resource_group_map = {
    "bicep": BICEP_RESOURCE_GROUP,
    "arm": ARM_RESOURCE_GROUP,
}

template_kind = INFRA_TEMPLATE_KIND.lower().strip()
if template_kind not in template_file_map:
    raise ValueError(f"Unsupported INFRA_TEMPLATE_KIND: {INFRA_TEMPLATE_KIND}. Use 'bicep' or 'arm'.")

TEMPLATE_FILE = template_file_map[template_kind]
RESOURCE_GROUP = resource_group_map[template_kind]

print(f"Bicep RG         : {BICEP_RESOURCE_GROUP}")
print(f"ARM RG           : {ARM_RESOURCE_GROUP}")
print(f"Active RG        : {RESOURCE_GROUP}")
print(f"Location         : {LOCATION}")
print(f"Demo ID          : {DEMO_ID}")
print(f"Deployment name  : {DEPLOYMENT_RUN_NAME}")
print(f"Template kind    : {template_kind}")
print(f"Template file    : {TEMPLATE_FILE}")

## Step 2 – Create the resource group

In [ ]:
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

group_result = subprocess.run(
    [
        az_cmd,
        "group",
        "create",
        "--name",
        RESOURCE_GROUP,
        "--location",
        LOCATION,
        "--output",
        "table",
    ],
check=True,
 )

print("Resource group created or already exists")

## Step 3 - Deploy infrastructure template

Use `INFRA_TEMPLATE_KIND` from Step 1 to choose Bicep (`infra/main.bicep`) or ARM JSON (`infra/main.json`).

In [ ]:
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

template_path = Path(TEMPLATE_FILE)
if not template_path.exists():
    raise FileNotFoundError(f"Template file not found: {template_path.resolve()}")

deploy_result = subprocess.run(
    [
        az_cmd,
        "deployment",
        "group",
        "create",
        "--name",
        DEPLOYMENT_RUN_NAME,
        "--resource-group",
        RESOURCE_GROUP,
        "--template-file",
        str(template_path),
        "--parameters",
        f"location={LOCATION}",
        f"demoId={DEMO_ID}",
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=False,
)

if deploy_result.returncode != 0:
    print(f"Deployment failed for {template_kind} template:")
    print(f"\nstderr:\n{deploy_result.stderr}")
    print(f"\nstdout:\n{deploy_result.stdout}")
    raise RuntimeError(f"Deployment failed with exit code {deploy_result.returncode}")

print(f"Deployment '{DEPLOYMENT_RUN_NAME}' succeeded using {template_kind} template")

## Step 4 - Retrieve outputs from deployed resources

Read account metadata and validate Entra token auth for downstream notebooks.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from azure.identity import AzureCliCredential

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

sub_result = subprocess.run(
    [az_cmd, "account", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
    check=False,
)
subscription_id = sub_result.stdout.strip()
if sub_result.returncode != 0 or not subscription_id:
    raise RuntimeError(
        "Failed to resolve active Azure subscription from Azure CLI.\n"
        f"stderr: {sub_result.stderr.strip()}\n"
        f"stdout: {sub_result.stdout.strip()}"
    )

outputs = json.loads(deploy_result.stdout).get("properties", {}).get("outputs", {})
account_name = outputs.get("aiServicesAccountName", {}).get("value", "")
ai_endpoint = outputs.get("aiServicesEndpoint", {}).get("value", "")
deployment_name = outputs.get("aiDeploymentName", {}).get("value", "")
foundry_project_name = outputs.get("foundryProjectName", {}).get("value", "")
foundry_project_endpoint = outputs.get("foundryProjectEndpoint", {}).get("value", "")

if not account_name or not ai_endpoint or not deployment_name:
    raise RuntimeError("Could not read aiServicesAccountName/aiServicesEndpoint/aiDeploymentName from deployment outputs.")

auth_result = subprocess.run(
    [
        az_cmd,
        "cognitiveservices",
        "account",
        "show",
        "--name",
        account_name,
        "--resource-group",
        RESOURCE_GROUP,
        "--subscription",
        subscription_id,
        "--query",
        "properties.disableLocalAuth",
        "--output",
        "tsv",
    ],
    capture_output=True,
    text=True,
    check=False,
)
if auth_result.returncode != 0:
    raise RuntimeError(
        "Failed to read AI Services auth mode from Azure CLI.\n"
        f"account: {account_name}\n"
        f"resource_group: {RESOURCE_GROUP}\n"
        f"subscription: {subscription_id}\n"
        f"stderr: {auth_result.stderr.strip()}\n"
        f"stdout: {auth_result.stdout.strip()}"
    )

disable_local_auth = auth_result.stdout.strip().lower() == "true"

credential = AzureCliCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "AzureCliCredential"
aad_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
if not aad_token:
    raise RuntimeError("Failed to acquire Azure AD token for Azure AI Services.")

auth_mode = "aad"

print(f"Subscription      : {subscription_id}")
print(f"Account name      : {account_name}")
print(f"AI endpoint       : {ai_endpoint}")
print(f"Deployment        : {deployment_name}")
print(f"Project name      : {foundry_project_name}")
print(f"Project endpoint  : {foundry_project_endpoint}")
print(f"disableLocalAuth  : {disable_local_auth}")
print(f"Auth mode         : {auth_mode}")
print(f"Credential source : {credential_source}")
print(f"Token preview     : {aad_token[:12]}...{aad_token[-10:]}")

## Step 5 - Write outputs to env/.env

This writes the resolved deployment values and notebook defaults to `env/.env`.

> Security note: `env/.env` is git-ignored. Never commit it.

In [ ]:
import pathlib

env_content = f"""# Auto-generated by 01_deploy_infra.ipynb - do not commit
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RESOURCE_GROUP}
AZURE_LOCATION={LOCATION}
AZURE_ACCOUNT_NAME={account_name}
AZURE_AI_ENDPOINT={ai_endpoint}
AZURE_AUTH_MODE={auth_mode}
AZURE_AI_DEPLOYMENT={deployment_name}
AZURE_FOUNDRY_PROJECT_NAME={foundry_project_name}
AZURE_FOUNDRY_PROJECT_ENDPOINT={foundry_project_endpoint}
AZURE_SPEECH_VOICE=en-US-JennyNeural
LOCAL_IMAGE_PATH=./data/sample-image.png
LOCAL_AUDIO_PATH=./data/sample-audio.wav
"""

env_path = pathlib.Path("../env/.env")
env_path.write_text(env_content, encoding="utf-8")
print(f"Written: {env_path.resolve()}")

## Step 6 - Verify `env/.env`

Confirm the generated environment file exists and contains expected keys before moving on.

In [ ]:
import pathlib

env_path = pathlib.Path("../env/.env")
example_path = pathlib.Path("../env/.env.example")

if env_path.exists():
    print("env/.env found")
    print(f"Path: {env_path.resolve()}")
else:
    print("ERROR: env/.env not found")
    print("Step 5 should generate this file.")
    print(f"Expected variable contract from {example_path}:")
    print(example_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(env_path)

---

## Infrastructure deployed

Move to **02_configure.ipynb** to verify the generated environment file and deployment defaults.

---

## Tear-down

In [ ]:
# Uncomment and run to delete all resources when finished.
# !az group delete --name {RESOURCE_GROUP} --yes --no-wait
# print("Resource group deletion triggered.")